# 📦 Notebook 01 — Data Collection
## Sales Data Analytics | Superstore Sales Dataset

---

### 🎯 Objective
This notebook covers the **Data Collection** phase of the analytics pipeline:
- Load the Superstore Sales dataset
- Understand the data structure and schema
- Perform initial profiling and sanity checks
- Document dataset characteristics

### 📋 Dataset
| Attribute | Detail |
|---|---|
| **Name** | Superstore Sales Dataset |
| **Source** | Tableau Public (Sample Superstore) |
| **Period** | 2014 – 2017 |
| **Records** | ~9,994 rows |
| **Columns** | 21 |

---

In [ ]:
# ── Library Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.width', 200)

print('✅ Libraries loaded successfully')

## 1.1 — Load Dataset

In [ ]:
# ── Load the dataset ─────────────────────────────────────────────────────────
DATA_PATH = Path('../dataset/superstore_sales.csv')

df = pd.read_csv(DATA_PATH)

print(f'✅ Dataset loaded successfully!')
print(f'   Rows    : {df.shape[0]:,}')
print(f'   Columns : {df.shape[1]}')
print(f'   Memory  : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

## 1.2 — First Look at the Data

In [ ]:
# First 5 rows
print('── First 5 Rows ─────────────────────────────────────────────────────────')
df.head()

In [ ]:
# Last 5 rows
print('── Last 5 Rows ──────────────────────────────────────────────────────────')
df.tail()

In [ ]:
# Random sample — check variety
print('── Random Sample (5 rows) ───────────────────────────────────────────────')
df.sample(5, random_state=42)

## 1.3 — Schema & Data Types

In [ ]:
# Column names, types, and non-null counts
print('── Data Types & Non-Null Counts ─────────────────────────────────────────')
df.info()

In [ ]:
# Columns summary table
schema = pd.DataFrame({
    'Column'    : df.columns,
    'Dtype'     : df.dtypes.values,
    'Non-Null'  : df.notnull().sum().values,
    'Null Count': df.isnull().sum().values,
    'Unique'    : df.nunique().values,
    'Sample'    : [df[col].dropna().iloc[0] if df[col].notnull().any() else None for col in df.columns]
})
schema

## 1.4 — Statistical Summary

In [ ]:
# Numeric columns statistical summary
print('── Numeric Columns — Descriptive Statistics ─────────────────────────────')
df.describe().round(2)

In [ ]:
# Categorical columns summary
print('── Categorical Columns — Unique Values ──────────────────────────────────')
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    unique_vals = df[col].nunique()
    top_val = df[col].value_counts().index[0]
    print(f'  {col:<20} | {unique_vals:>5} unique | Top: {top_val}')

## 1.5 — Unique Value Distributions

In [ ]:
# Distribution of key categorical columns
for col in ['Category', 'Sub-Category', 'Region', 'Segment', 'Ship Mode']:
    print(f'\n── {col} ──')
    print(df[col].value_counts())

## 1.6 — Date Range & Time Span

In [ ]:
# Convert dates
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

print(f'Order Date Range : {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
print(f'Ship Date Range  : {df["Ship Date"].min().date()} → {df["Ship Date"].max().date()}')
print(f'Time Span        : {(df["Order Date"].max() - df["Order Date"].min()).days} days (~{round((df["Order Date"].max() - df["Order Date"].min()).days/365, 1)} years)')
print(f'Years Covered    : {sorted(df["Order Date"].dt.year.unique())}')

## 1.7 — Missing Values Check

In [ ]:
# Missing values heatmap
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('✅ No missing values found in the dataset!')
else:
    print('⚠️  Missing values detected:')
    print(missing_df)

## 1.8 — Duplicate Check

In [ ]:
# Check for duplicate rows
dup_count = df.duplicated().sum()
print(f'Total duplicate rows  : {dup_count}')
print(f'Duplicate order IDs   : {df.duplicated(subset="Order ID").sum()}')
print(f'Unique Order IDs      : {df["Order ID"].nunique():,}')
print(f'Unique Customer IDs   : {df["Customer ID"].nunique():,}')
print(f'Unique Product IDs    : {df["Product ID"].nunique():,}')

## 1.9 — Key Observations

| # | Observation |
|---|---|
| 1 | Dataset covers **4 years** (2014–2017) of transactional data |
| 2 | **21 columns** covering orders, customers, products, and financials |
| 3 | Three product **categories**: Furniture, Office Supplies, Technology |
| 4 | Four sales **regions**: East, West, Central, South |
| 5 | Three customer **segments**: Consumer, Corporate, Home Office |
| 6 | `Profit` can be **negative** — indicates loss-making transactions |
| 7 | `Order Date` and `Ship Date` need datetime conversion |
| 8 | `Postal Code` should be treated as **string** to preserve leading zeros |

---

**➡️ Next Step**: Notebook 02 — Data Cleaning